In [ ]:
import gc

gc.collect()

# elevator-rmnd

Use the simulated dataset generated by `dataset.py` to train
regression models, in order to predict the remaining useful life (RUL)
of lifts.

**This approach uses classical ML techniques.**

In [1]:
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import joblib as jl
import seaborn as sns
import os

sns.set_theme(context="notebook", style="ticks")

MODEL_DIR = os.path.join(os.getcwd(), "models")

## preprocessing
* Import the simulated dataset
* ~~Convert lift ids to integers~~ [already handled by `dataset.py`]
* ~~Compute instantaneous time derivatives of each sensor metric~~ [already handled by `dataset.py`] 
* ~~Compute RUL in h~~ [already handled by `dataset.py`]
* Split data into training/testing sets
* Scale data with a `Scaler`

In [2]:
import ipywidgets as widgets
from IPython.display import display
uploader = widgets.FileUpload(accept='', multiple=True)
display(uploader)

FileUpload(value={}, description='Upload', multiple=True)

In [ ]:
if uploader.value:
    for filename, file_info in uploader.value.items():
        with open(filename, "wb") as f:
            f.write(file_info['content'])
        print(f"Saved: {filename}")
else:
    print("No files uploaded yet.")

In [ ]:
df_full = pd.read_pickle(uploaded["liftdata_v3.pkl"])
df_full.head(10)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

Since there exist rows that have `np.inf` as their predicted RUL, extract those rows for experimentation later on.
Retain the remaining roles (with valid RULs) for training and testing.

In [ ]:
df_full.dropna(inplace=True)  # Drop any records with missing values

df_experimental = df_full[
    df_full["RUL_hrs"] == np.inf
]  # Filter out records with no further maintenance
df_useful = df_full[
    df_full["RUL_hrs"] != np.inf
]  # Retain the remaining records for training/testing

# Check to see the experimental/useful split was done correctly
assert all(
    df_experimental["RUL_hrs"] == np.inf
), "All records in experimental set should have RUL_hrs == np.inf"
assert all(
    df_useful["RUL_hrs"] != np.inf
), "All records in useful set should have valid RUL_hrs"

df_useful.head(40)

Then perform a 85/15 train/test split on the remaining useful data.

In [ ]:
X = df_useful.drop(columns=["timestamp", "lift_model", "maintenance_done", "RUL_hrs"])
y = df_useful["RUL_hrs"]
X.dtypes, y.dtypes

In [ ]:
# Split df_useful into 85/15 train/test sets, preserving temporal contiguity per lift_id
train_dfs = []
test_dfs = []

for lift_id, group in df_useful.groupby('lift_id'):
    # Get the group's data
    group_data = group.reset_index(drop=True)
    n = len(group_data)
    
    # Calculate split indices for contiguous splits
    train_size = int(n * 0.85)
    
    # Split contiguously (preserves temporal order)
    train_dfs.append(group_data[:train_size])
    test_dfs.append(group_data[train_size:])

# Concatenate all splits
train_split = pd.concat(train_dfs, ignore_index=True)
test_split = pd.concat(test_dfs, ignore_index=True)

# Extract X and y from the splits
X_train = train_split.drop(columns=["timestamp", "lift_model", "maintenance_done", "RUL_hrs"])
y_train = train_split["RUL_hrs"]
X_test = test_split.drop(columns=["timestamp", "lift_model", "maintenance_done", "RUL_hrs"])
y_test = test_split["RUL_hrs"]

# Dummy checks
assert len(X_train) == len(y_train), "X_train and y_train should have the same size"
assert len(X_test) == len(y_test), "X_test and y_test should have the same size"
assert y_train.isna().sum() == 0, "y_train should not have any null values"
assert y_test.isna().sum() == 0, "y_test should not have any null values"

Use `StandardScaler` to transform the data (for classical ML techniques).

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## classical ML techniques
We experiment with a few classical regression techniques and evaluate them on a set of metrics.
Metrics of interest include:
* mean absolute error (`mae`): the mean deviation (in h) of predicted RUL from true RUL
* r2 score (`r2`): how well the true RUL and predicted RUL match up

In [ ]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.svm import SVR
from catboost import CatBoostRegressor
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestRegressor

In [ ]:
model_architectures = {
    "svr": SVR(kernel="rbf", epsilon=0.1),
    "xgb": XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.1, verbosity=0),
    "cat": CatBoostRegressor(
        n_estimators=500, max_depth=6, learning_rate=0.1, verbose=False
    ),
    "lgb": LGBMRegressor(n_estimators=500, max_depth=6, learning_rate=0.1, verbose=0),
    "rf": RandomForestRegressor(n_estimators=500, max_depth=6),
}

Just carry out model training.
* Record the RMSE, MAE and R2 of each model
* Record each model's $\hat{y}$ for visual comparison

In [ ]:
all_metrics = []
model_preds = pd.DataFrame(columns=["name", "y_test", "y_pred", "cv"])

for name, arch in model_architectures.items():
    model = arch
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    preds_df = pd.DataFrame(
        {
            "name": np.full(len(y_test), name),  # Broadcast name to all rows
            "y_test": y_test.values,
            "y_pred": y_pred,
            "cv": np.zeros(len(y_test), dtype=int),
        }
    )
    model_preds = pd.concat([model_preds, preds_df], ignore_index=True)

    # Compute metrics
    rmse = np.round(np.sqrt(np.sum((y_test - y_pred) ** 2)), 4)
    mae = np.round(np.abs(y_test - y_pred).mean(), 4)
    r2 = np.round(r2_score(y_test, y_pred), 4)
    scores = [
        {"name": name, "metric": "rmse", "value": rmse, "cv": 0},
        {"name": name, "metric": "mae", "value": mae, "cv": 0},
        {"name": name, "metric": "r2", "value": r2, "cv": 0},
    ]
    print(scores)
    all_metrics.extend(scores)

    # Export model
    fname = os.path.join(
        MODEL_DIR, f"{name}_{datetime.now().strftime('%Y%m%d-%H%M')}.joblib"
    )
    jl.dump(model, fname)

model_metrics = pd.DataFrame(all_metrics)

In [ ]:
model_metrics

In [ ]:
g = sns.FacetGrid(
    model_metrics, row="metric", sharex=False, height=3, aspect=2, hue="name"
)
g.map(sns.barplot, "value", "name", order=model_metrics["name"].unique())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
sns.scatterplot(
    data=model_preds,
    x="y_test",
    y="y_pred",
    hue="name",
    style="name",
    ax=ax,
)

## experimentation
We can evaluate which features the models thought were the most valuable (i.e. feature importance).

In [ ]:
import glob

In [ ]:
feature_importances = {
    "model": [],
    "feature": [],
    "importance_mean": [],
    "importance_std": [],
}

for name in model_architectures.keys():
    model_files = glob.glob(os.path.join(MODEL_DIR, f"{name}_*.joblib"))
    best_model = jl.load(max(model_files, key=os.path.getctime))
    
    r = permutation_importance(
        best_model, X_test, y_test, n_repeats=10, random_state=67, n_jobs=-1
    )
    for i in r.importances_mean.argsort()[::-1]:
        if r.importances_mean[i] - 2 * r.importances_std[i] > 0:
            feature_importances["model"].append(name)
            feature_importances["feature"].append(X.columns[i])
            feature_importances["importance_mean"].append(r.importances_mean[i])
            feature_importances["importance_std"].append(r.importances_std[i])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
ax.semilogx()
sns.barplot(
    data=feature_importances,
    x="importance_mean",
    y="feature",
    hue="model",
    orient="h",
    palette="Set2",
).set_title("Feature importances")

In [ ]:
trial = df_experimental[0].reshape(1, -1)
xgb_preds = best_xgb.predict(trial)
cat_preds = best_cat.predict(trial)
scaler.inverse_transform(trial), xgb_preds, cat_preds